# Phase 1 Final Comparator Runner

This notebook is an orchestration and audit surface only. Durable scientific logic lives in `src/` and is exercised through `python -m src.cli phase1_final_comparator_runner`.

Integrity boundary:

- Input must be the reviewed `final_feature_matrix.csv` from notebook 23.
- The feature matrix contains feature values and labels only; it must not contain logits, metrics, model outputs, controls, calibration, influence, or runtime leakage logs.
- A2d Riemannian must be blocked unless valid final covariance/tangent inputs exist. Do not approximate A2d from the bandpower feature matrix.
- Claims remain closed even after this runner writes comparator outputs.
- Do not interpret BA/ECE/Brier diagnostics as efficacy evidence until the full preregistered comparator, controls, calibration, influence, and reporting package passes.

In [ ]:
# Cell 1 - Bootstrap repo, pull current code, and run unit tests.

from __future__ import annotations

import getpass
import hashlib
import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
RUN_UNITTESTS = True

def run(cmd, cwd=None, check=True):
    printable = ['<redacted>' if 'Authorization:' in str(part) else str(part) for part in cmd]
    print('$', ' '.join(printable))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

if not REPO_DIR.exists():
    token = getpass.getpass('Enter GitHub token for private repo, or leave blank for public clone: ')
    if token:
        header = 'Authorization: Basic ' + __import__('base64').b64encode(f'x-access-token:{token}'.encode()).decode()
        run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
    else:
        run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

print('Repo:', REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)

if RUN_UNITTESTS:
    unit = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, text=True)
    if unit.returncode != 0:
        raise subprocess.CalledProcessError(unit.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])

In [ ]:
# Cell 2 - Select reviewed source artifacts and keep launch disabled by default.

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'

PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'
FEATURE_MATRIX_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_feature_matrix'
RUNNER_READINESS_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_comparator_runner_readiness'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_comparator_runner'
PLAN_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_comparator_runner_plan'

RUN_FINAL_COMPARATOR_RUNNER = False
REQUIRED_ACK = 'I understand this final comparator runner is claim-closed and must not be interpreted as efficacy evidence'
MANUAL_LAUNCH_ACK = ''

def latest_run(root: Path) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        return Path(latest.read_text(encoding='utf-8').strip())
    runs = sorted(path for path in root.iterdir() if path.is_dir())
    if not runs:
        raise FileNotFoundError(f'No run dirs found under {root}')
    return runs[-1]

FEATURE_MATRIX_RUN = latest_run(FEATURE_MATRIX_ROOT)
RUNNER_READINESS_RUN = latest_run(RUNNER_READINESS_ROOT)

print('Prereg bundle:', PREREG_BUNDLE)
print('Feature matrix run:', FEATURE_MATRIX_RUN)
print('Runner readiness run:', RUNNER_READINESS_RUN)
print('Output root:', OUTPUT_ROOT)
print('Run flag:', RUN_FINAL_COMPARATOR_RUNNER)

In [ ]:
# Cell 3 - Validate source boundaries before any launch.

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Prereg identity hash mismatch.'

matrix_summary = read_json(FEATURE_MATRIX_RUN / 'phase1_final_feature_matrix_summary.json')
matrix_validation = read_json(FEATURE_MATRIX_RUN / 'phase1_final_feature_matrix_validation.json')
matrix_schema = read_json(FEATURE_MATRIX_RUN / 'phase1_final_feature_matrix_schema.json')
matrix_claim_state = read_json(FEATURE_MATRIX_RUN / 'phase1_final_feature_matrix_claim_state.json')
matrix_csv = Path(matrix_summary.get('matrix_path') or FEATURE_MATRIX_RUN / 'final_feature_matrix.csv')

print('Feature matrix status:', matrix_summary.get('status'))
print('Feature matrix ready:', matrix_summary.get('feature_matrix_ready'))
print('Feature matrix rows:', matrix_summary.get('n_rows'))
print('Feature matrix features:', matrix_summary.get('n_features'))
print('Feature matrix CSV:', matrix_csv)

assert matrix_summary.get('status') == 'phase1_final_feature_matrix_materialized'
assert matrix_summary.get('feature_matrix_ready') is True
assert matrix_validation.get('status') == 'phase1_final_feature_matrix_validation_passed'
assert matrix_summary.get('nonfinite_feature_values') == 0
assert matrix_summary.get('contains_model_outputs') is False
assert matrix_summary.get('contains_logits') is False
assert matrix_summary.get('contains_metrics') is False
assert matrix_schema.get('contains_model_outputs') is False
assert matrix_schema.get('contains_logits') is False
assert matrix_schema.get('contains_metrics') is False
assert matrix_claim_state.get('claim_ready') is False
assert matrix_csv.exists(), f'Missing final_feature_matrix.csv: {matrix_csv}'

readiness_summary = read_json(RUNNER_READINESS_RUN / 'phase1_final_comparator_runner_readiness_summary.json')
readiness_claim_state = read_json(RUNNER_READINESS_RUN / 'phase1_final_comparator_runner_claim_state.json')
print('Runner readiness status:', readiness_summary.get('status'))
print('Upstream manifests ready:', readiness_summary.get('upstream_manifests_ready'))
print('Final comparator outputs present before run:', readiness_summary.get('final_comparator_outputs_present'))
assert readiness_summary.get('status') == 'phase1_final_comparator_runner_readiness_recorded'
assert readiness_summary.get('upstream_manifests_ready') is True
assert readiness_summary.get('final_comparator_outputs_present') is False
assert readiness_summary.get('runtime_comparator_logs_audited') is False
assert readiness_summary.get('smoke_artifacts_promoted') is False
assert readiness_claim_state.get('claim_ready') is False

In [ ]:
# Cell 4 - Record a launch plan with explicit non-claim boundary.

created_utc = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
PLAN_DIR = PLAN_ROOT / created_utc
PLAN_DIR.mkdir(parents=True, exist_ok=False)

help_result = subprocess.run(
    ['python', '-m', 'src.cli', 'phase1_final_comparator_runner', '--help'],
    cwd=REPO_DIR,
    text=True,
    capture_output=True,
)
runner_available = help_result.returncode == 0 and '--feature-matrix-run' in help_result.stdout

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_comparator_runner',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--feature-matrix-run', str(FEATURE_MATRIX_RUN),
    '--runner-readiness-run', str(RUNNER_READINESS_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--runner-config', 'configs/phase1/final_comparator_runner.json',
]

plan = {
    'status': 'phase1_final_comparator_runner_plan_recorded',
    'created_utc': created_utc,
    'runner_available': runner_available,
    'run_flag': RUN_FINAL_COMPARATOR_RUNNER,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'feature_matrix_run': str(FEATURE_MATRIX_RUN),
    'feature_matrix_csv': str(matrix_csv),
    'feature_matrix_rows': matrix_summary.get('n_rows'),
    'feature_matrix_features': matrix_summary.get('n_features'),
    'runner_readiness_run': str(RUNNER_READINESS_RUN),
    'planned_command': cmd,
    'scientific_integrity_boundary': {
        'claim_ready': False,
        'headline_phase1_claim_open': False,
        'smoke_artifacts_promoted': False,
        'a2d_must_block_without_covariance_or_tangent_input': True,
        'not_ok_to_claim': [
            'decoder efficacy',
            'A3 distillation efficacy',
            'A4 privileged-transfer efficacy',
            'A4 superiority',
            'full Phase 1 neural comparator performance',
        ],
    },
}
(PLAN_DIR / 'phase1_final_comparator_runner_plan.json').write_text(json.dumps(plan, indent=2) + '\n', encoding='utf-8')
print(json.dumps(plan, indent=2))

In [ ]:
# Cell 5 - Manual hold or explicit launch through CLI.

if not runner_available:
    raise RuntimeError('CLI route phase1_final_comparator_runner is unavailable. Pull the correct commit before continuing.')

if not RUN_FINAL_COMPARATOR_RUNNER:
    hold = {
        'status': 'phase1_final_comparator_runner_manual_hold',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'plan_dir': str(PLAN_DIR),
        'run_flag': RUN_FINAL_COMPARATOR_RUNNER,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
        'next': 'Review the plan. Rerun only after setting RUN_FINAL_COMPARATOR_RUNNER=True and the exact acknowledgement.',
    }
    (PLAN_DIR / 'phase1_final_comparator_runner_manual_hold.json').write_text(json.dumps(hold, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(hold, indent=2))
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch final comparator runner without explicit non-claim acknowledgement.')
else:
    run(cmd, cwd=REPO_DIR)
    print('Final comparator runner command completed. Review artifacts before any interpretation.')

In [ ]:
# Cell 6 - Inspect latest runner output if the command was launched.

latest_output = None
summary = None
if RUN_FINAL_COMPARATOR_RUNNER:
    latest_output = latest_run(OUTPUT_ROOT)
    print('Latest final comparator runner output:', latest_output)
    required = [
        'phase1_final_comparator_runner_inputs.json',
        'phase1_final_comparator_runner_summary.json',
        'phase1_final_comparator_runner_report.md',
        'phase1_final_comparator_runner_source_links.json',
        'phase1_final_comparator_runner_input_validation.json',
        'phase1_final_comparator_runtime_leakage_audit.json',
        'phase1_final_comparator_completeness_table.json',
        'phase1_final_comparator_runner_claim_state.json',
    ]
    for name in required:
        print(name, 'exists =', (latest_output / name).exists())
    summary = read_json(latest_output / 'phase1_final_comparator_runner_summary.json')
    print(json.dumps(summary, indent=2))
else:
    print('Manual hold active. No runner output inspected in this pass.')

In [ ]:
# Cell 7 - Assertions for launched run.

if RUN_FINAL_COMPARATOR_RUNNER:
    assert summary is not None
    assert summary.get('status') in {
        'phase1_final_comparator_runner_partial_with_blockers',
        'phase1_final_comparator_runner_complete_claim_closed',
    }
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('full_phase1_claim_bearing_run_allowed') is False
    assert summary.get('smoke_artifacts_promoted') is False
    assert summary.get('feature_matrix_path') == str(matrix_csv)
    if 'A2d_riemannian' in summary.get('blocked_comparators', []):
        print('A2d is blocked as expected when only final_feature_matrix.csv is available.')
        assert summary.get('final_comparator_outputs_present') is False
    claim_state = read_json(latest_output / 'phase1_final_comparator_runner_claim_state.json')
    assert claim_state.get('claim_ready') is False
    assert claim_state.get('headline_phase1_claim_open') is False
    assert 'full Phase 1 neural comparator performance' in claim_state.get('not_ok_to_claim', [])
else:
    print('Assertions skipped because run flag is False.')

In [ ]:
# Cell 8 - Write non-claim review note only after launched run.

if RUN_FINAL_COMPARATOR_RUNNER:
    completeness = read_json(latest_output / 'phase1_final_comparator_completeness_table.json')
    leakage = read_json(latest_output / 'phase1_final_comparator_runtime_leakage_audit.json')
    claim_state = read_json(latest_output / 'phase1_final_comparator_runner_claim_state.json')
    review = {
        'status': 'phase1_final_comparator_runner_review_pass_non_claim_with_recorded_blockers',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'reviewed_run': str(latest_output),
        'completed_comparators': summary.get('completed_comparators'),
        'blocked_comparators': summary.get('blocked_comparators'),
        'checks_passed': [
            'required_artifacts_present',
            'feature_matrix_used_as_only_model_input',
            'feature_matrix_contains_no_logits_metrics_or_model_outputs',
            'runtime_leakage_logs_written_for_completed_comparators',
            'outer_test_subject_not_used_for_completed_comparator_fit',
            'claim_ready_false',
            'headline_claim_closed',
            'smoke_artifacts_not_promoted',
        ],
        'a2d_handling': 'blocked unless final covariance/tangent inputs are implemented; no proxy A2d result is accepted from bandpower feature matrix alone',
        'runtime_leakage_summary': {
            'outer_test_subject_used_for_any_fit': leakage.get('outer_test_subject_used_for_any_fit'),
            'runtime_logs_audited_for_all_required_comparators': leakage.get('runtime_logs_audited_for_all_required_comparators'),
        },
        'completeness_status': completeness.get('status'),
        'metrics_interpretation': {
            'allowed': 'Implementation diagnostics and artifact completeness review only.',
            'not_allowed': 'Do not use any metric as efficacy, superiority, or privileged-transfer evidence.',
        },
        'claim_blockers': claim_state.get('blockers'),
        'not_ok_to_claim': claim_state.get('not_ok_to_claim'),
    }
    note_path = latest_output / 'phase1_final_comparator_runner_review_note.json'
    note_path.write_text(json.dumps(review, indent=2) + '\n', encoding='utf-8')
    print('Review note written:', note_path)
    print(json.dumps(review, indent=2))
else:
    print('Review note not written because run flag is False.')

In [ ]:
# Cell 9 - Closeout.

print('================ Phase 1 Final Comparator Runner Closeout ================')
print('Notebook fix marker: phase1_final_comparator_runner_v1_20260421')
print('Plan source of truth:', PLAN_DIR)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_COMPARATOR_RUNNER)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
if RUN_FINAL_COMPARATOR_RUNNER:
    print('Latest final comparator runner output:', latest_output)
    print('Completed comparators:', summary.get('completed_comparators'))
    print('Blocked comparators:', summary.get('blocked_comparators'))
    print('Claim blockers:', summary.get('claim_blockers'))
    print('CHECK REQUIRED: Review comparator manifests, logits, subject metrics and runtime leakage logs before downstream governance.')
else:
    print('HELD: Runner appears available, but execution was not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun only with explicit non-claim acknowledgement if appropriate.')
print('\nNOT OK TO CLAIM: This notebook does not prove decoder efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')